In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import joblib
from pathlib import Path

# 1. Charger les données
df = pd.read_csv('../data/customer_churn_business_dataset.csv') 

# 2. Appliquer la décision de l'EDA : suppression du bruit géopolitique
df = df.drop(columns=['country', 'customer_id']) # On enlève aussi l'ID qui n'a pas de valeur prédictive

# Définir notre matrice de features X et notre cible y
X = df.drop(columns=['churn'])
y = df['churn']

In [2]:
# Train-Test Split (80% Train / 20% Test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.20, 
    stratify=y, 
    random_state=42
)

print(f"Entraînement : {X_train.shape[0]} clients")
print(f"Validation   : {X_test.shape[0]} clients")

Entraînement : 8000 clients
Validation   : 2000 clients


In [3]:
# Identifier les types de colonnes (sans 'country')
numerical_features = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X_train.select_dtypes(include=['object']).columns.tolist()

# 1. Pipeline pour les variables numériques
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# 2. Pipeline pour les variables textuelles
cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='No_Complaint')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# 3. Assemblage global via ColumnTransformer
preprocessor = ColumnTransformer(transformers=[
    ('num', num_transformer, numerical_features),
    ('cat', cat_transformer, categorical_features)
])

C:\Users\Pret Jeff\AppData\Local\Temp\ipykernel_15984\2893924778.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X_train.select_dtypes(include=['object']).columns.tolist()


In [4]:
# Le préprocesseur apprend et transforme le Train
X_train_proc = preprocessor.fit_transform(X_train)

# Le préprocesseur transforme le Test (SANS réapprendre !)
X_test_proc = preprocessor.transform(X_test)

# Récupérer proprement le nom des nouvelles colonnes après encodage (très utile pour SHAP plus tard)
cat_encoder = preprocessor.named_transformers_['cat'].named_steps['onehot']
encoded_cat_features = cat_encoder.get_feature_names_out(categorical_features).tolist()
feature_names = numerical_features + encoded_cat_features

print(f"Forme finale des données après encodage : {X_train_proc.shape[1]} caractéristiques.")

Forme finale des données après encodage : 51 caractéristiques.


In [5]:
# Créer le dossier s'il n'existe pas
Path('../new_models').mkdir(exist_ok=True)

# Sauvegarder le préprocesseur et les jeux de données
joblib.dump(preprocessor, '../new_models/preprocessor_churn.joblib')

joblib.dump({
    'X_train': X_train_proc, 
    'X_test': X_test_proc,
    'y_train': y_train, 
    'y_test': y_test,
    'feature_names': feature_names
}, '../new_models/processed_data.joblib')

print("Fichiers sauvegardés avec succès ! Prêt pour le Notebook 03.")

Fichiers sauvegardés avec succès ! Prêt pour le Notebook 03.
